# Split-Protokoll: 80/20

**Ticket:** #135 – Train-/Validierungs-Split flexibel konfigurieren

**Hypothese:** Mehr Trainingsdaten können die kombinierte SVM-Referenz verbessern. Die Kennzahlen sind jedoch nur innerhalb desselben Split-Protokolls vergleichbar.

Vor der Ausführung dieses Notebooks im Terminal das 80/20-Artefakt erzeugen:

```bash
make data DATA_ARGS="--validation-size 0.2"
```

## 1. Branch und Ticket-Kontext prüfen

Die Implementierung gehört zu `data/135-flexible-split`. Ticket #135 verlangt eine zentral validierte Split-Konfiguration, einen reproduzierbaren Aufruf von `make data`, unveränderte Stratifikation nach Crop/Stage sowie eine Dokumentation der eingeschränkten Vergleichbarkeit verschiedener Split-Anteile.

## 2. Projektstruktur und bestehende Implementierung analysieren

Der Daten-Builder speichert die Zuordnung in `data/processed/split.csv` und die verwendete `SplitConfig` in `data/processed/split_config.json`. `load_split()` und `load_folds()` lesen diese gemeinsamen Artefakte; die Trainingsdaten werden hier nicht selbst neu gesplittet.

## 3. Ticket 1 implementieren

`SplitConfig(validation_size=...)` validiert den Anteil strikt zwischen 0 und 1. Der Builder erhält den Wert über `--validation-size`, erzeugt Split und CV-Folds deterministisch mit dem Projekt-Seed und protokolliert den Anteil als `split.validation_size` in MLflow.

## 4. Implementierung mit Tests validieren

Die Unit-Tests prüfen einen konfigurierten 80/20-Split und ungültige Anteile. Ausführen:

```bash
uv run python -m unittest tests.test_split -v
```

## 5. Experiment-Notebook einrichten

Die folgenden Zellen verwenden ausschließlich den erzeugten gemeinsamen 80/20-Split. Der Seed wird von den Datenartefakten und dem verwendeten Modell über `awp2.config.SEED` gesteuert.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from awp2.data import load_split, load_split_config
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_combined_svm
from awp2.preprocessing import PreprocessingConfig

### Split prüfen

`0.2` bedeutet 80 % Training und 20 % Validierung. Das Konfigurationsartefakt ist die verbindliche Quelle für den verwendeten Anteil.

In [ ]:
config = load_split_config()
split = load_split()

assert config.validation_size == 0.2, (
    "Erzeuge zuerst das Artefakt mit: make data DATA_ARGS=\"--validation-size 0.2\""
)

summary = pd.DataFrame(
    {
        "validation_size": [config.validation_size],
        "train_rows": [len(split.X_train)],
        "validation_rows": [len(split.X_val)],
        "train_fraction": [len(split.X_train) / (len(split.X_train) + len(split.X_val))],
        "validation_fraction": [len(split.X_val) / (len(split.X_train) + len(split.X_val))],
    }
)
display(summary)

## 6. Experiment ausführen und Ergebnisse prüfen

### Referenzmodell auf dem 80/20-Protokoll suchen

Die Referenz nutzt die bisher stärkste Preprocessing-Konfiguration: standardisierte Rohspektren sowie AEZ und zyklischen Month als Kontext. Vegetationsindizes, SNV und PCA wurden im bisherigen 70/30-Protokoll nicht übernommen. Die Kandidaten werden ausschließlich nach der Cross-Validation auf dem Trainingsanteil ausgewählt; die 20-%-Validierung bleibt bis zur abschließenden Bewertung unberührt.

In [ ]:
preprocessing = PreprocessingConfig(
    use_meta=True,
    use_cyclic_month=True,
    use_spectral_standard_scale=True,
)
tuned = tune(
    make_combined_svm(),
    TuneConfig(
        name="combined_svm_split_80_20_search",
        approach="combined",
        description=(
            "RBF-SVM mit skalierten Rohspektren sowie AEZ und zyklischem Month unter dem "
            "80/20-Split; C und gamma per CV."
        ),
        experiment="split-80-20",
        preprocessing=preprocessing,
        param_grid={
            "estimator__C": [30.0, 100.0, 300.0],
            "estimator__gamma": [0.001, 0.003, 0.01],
        },
    ),
)
print(f"MLflow tuning run: {tuned.run_id}")
display(tuned.candidates.sort_values("rank"))

### Einmal auf der 20-%-Validierung bewerten

Die Validierungsmetriken werden berichtet, aber nicht zur Auswahl von Hyperparametern verwendet.

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="combined_svm_split_80_20",
        approach="combined",
        description="Bester RBF-SVM-Kandidat unter dem 80/20-Split.",
        experiment="split-80-20",
        preprocessing=preprocessing,
        tuning_run=tuned.run_id,
    ),
)
print(f"MLflow evaluation run: {result.run_id}")
display(pd.DataFrame([result.metrics.model_dump()]).T)

### Fehleranalyse

Die Kennzahlen allein zeigen nicht, ob Fehler von der Kultur oder nur vom Stadium kommen. Die folgenden Tabellen trennen beides und zeigen die schwächsten Crop/Stage-Paare sowie die häufigsten Fehlzuordnungen.

In [ ]:
from awp2.data import combined_label
from awp2.evaluation import error_breakdown, recall_by_class

pairs = pd.DataFrame(
    {
        "true": combined_label(result.y_val),
        "predicted": combined_label(result.y_pred),
    }
)
top_confusions = (
    pairs.loc[pairs["true"] != pairs["predicted"]]
    .value_counts()
    .rename("count")
    .reset_index()
    .head()
)

display(pd.DataFrame([error_breakdown(result.y_val, result.y_pred).model_dump()]).T)
display(recall_by_class(result.y_val, result.y_pred).head())
display(top_confusions)

### Ergebnis festhalten

Im Experiment-Log beide MLflow-Run-IDs und `split.validation_size = 0.2` eintragen. Numerische Scores nicht direkt mit 70/30-Läufen vergleichen, weil sich sowohl Trainings- als auch Validierungsdaten unterscheiden.